# Multi-Level Monte Carlo: Results and Comparisons

**Based on Amelie's original visualisation scripts**

This notebook provides comprehensive analysis and visualisation of Multi-Level Monte Carlo (MLMC) results for American basket option pricing.

## Contents

1. **Setup and Configuration** - Problem parameters and utilities
2. **Volatility Surface Visualisation** - 3D plots of fitted local volatility
3. **Projection Error Analysis** - L² approximation quality
4. **Fine/Coarse Coupling Diagnostics** - Correlation scatter plots
5. **Single-Level vs Multi-Level Comparison** - Performance analysis
6. **Optimal Transport Validation** - OT map quality checks

---

## 1. Setup and Configuration

In [ ]:
# Standard imports
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
import os
import math

# Multi-Level imports
from ML_level_utilities import (
    GBM_paths, tot_degree_poly, normaleq_components_ML, 
    fit_local_vol, make_b_bar
)
from ML_telescoping_sum import scalings_l0, mlmc_l, make_c
from ML_optimal_transport import (
    GaussianBrenierMap, logpaths_maps, apply_maps, 
    validate_maps, mlmc_l_OT, make_c_OT
)

# Create plot directories
os.makedirs("plots/VolSurf", exist_ok=True)
os.makedirs("plots/VolSurfOT", exist_ok=True)
os.makedirs("plots/PairScatter", exist_ok=True)
os.makedirs("plots/Comparison", exist_ok=True)

# Plotting configuration
plt.rcParams['figure.figsize'] = (10, 7)
plt.rcParams['font.size'] = 11
plt.rcParams['lines.linewidth'] = 1.5

print("✓ All imports successful")
print("✓ Plot directories created")

### Problem Parameters

In [ ]:
# Basket parameters
d = 3  # Number of assets
P1 = np.ones(d) / d  # Equal-weighted basket
r = 0.05  # Risk-free rate
T = 1.0  # Maturity

# Initial prices (linearly spaced around 250)
x0 = np.linspace(225, 275, num=d)[:, np.newaxis]

# Volatilities (decreasing)
vol = np.array([0.2, 0.15, 0.1])

# Correlation matrix (realistic structure)
cov_mat = np.array([
    [1.0, 0.8, 0.3],
    [0.8, 1.0, 0.1],
    [0.3, 0.1, 1.0]
])

# MLMC parameters
h0 = 0.125  # Coarsest timestep
max_deg = 3  # Maximum polynomial degree (determines number of levels)
C = 80  # Sample size scaling factor

print("Problem Configuration:")
print(f"  Dimension: d = {d}")
print(f"  Initial prices: {x0.flatten()}")
print(f"  Volatilities: {vol}")
print(f"  Risk-free rate: r = {r}")
print(f"  Maturity: T = {T}")
print(f"  Max polynomial degree: {max_deg}")
print(f"  Number of MLMC levels: {max_deg + 1}")

---

## 2. Volatility Surface Visualisation

**Based on `ML_volsurf.py`**

Creates 3D wireframe plots of the fitted local volatility surface $\bar{b}(t, S)$.

In [ ]:
def plot_volatility_surface(b_bar, s_min, s_max, T, title="Volatility Surface",
                            save_path=None, show_plot=True):
    """Create 3D wireframe plot of local volatility surface."""
    # Create grid
    t_vals = np.linspace(0, T, 50)
    s_vals = np.linspace(s_min, s_max, 50)
    T_grid, S_grid = np.meshgrid(t_vals, s_vals)
    
    # Evaluate volatility surface
    B_grid = b_bar(T_grid, S_grid)
    
    # Create 3D plot
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    surf = ax.plot_wireframe(T_grid, S_grid, B_grid, 
                             color='steelblue', alpha=0.6,
                             linewidth=0.8, antialiased=True)
    
    ax.set_xlabel('Time $t$', fontsize=12, labelpad=10)
    ax.set_ylabel('Basket Value $S$', fontsize=12, labelpad=10)
    ax.set_zlabel(r'Local Volatility $\bar{b}(t,S)$', fontsize=12, labelpad=10)
    ax.set_title(title, fontsize=14, pad=20)
    ax.view_init(elev=20, azim=45)
    ax.grid(True, alpha=0.3)
    
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
        print(f"✓ Saved: {save_path}")
    
    if show_plot:
        plt.show()
    else:
        plt.close()
    
    print(f"  Min: {B_grid.min():.6f}, Max: {B_grid.max():.6f}, Mean: {B_grid.mean():.6f}")
    if B_grid.min() < 0:
        print(f"    ⚠️  WARNING: Negative volatilities detected!")
    else:
        print(f"    ✓ All volatilities positive")

In [ ]:
print("Running pilot simulation...")
s_min0, s_max0 = scalings_l0(x0, T, h0, r, cov_mat, vol, max_deg, P1, M_0=10000)
print(f"✓ Domain: [{s_min0:.2f}, {s_max0:.2f}]")

print("\nRunning MLMC...")
c_mlmc = make_c(x0, T, h0, r, cov_mat, vol, max_deg, P1, s_min0, s_max0, C)
pairs = tot_degree_poly(max_deg)
b_bar_mlmc = make_b_bar(c_mlmc, pairs, s_min0, s_max0, T, max_deg)

plot_volatility_surface(b_bar_mlmc, s_min0, s_max0, T,
                        title=f"MLMC Volatility Surface (max_deg={max_deg})",
                        save_path=f"plots/VolSurf/VolSurf_maxdeg{max_deg}.pdf")

In [ ]:
print("\nRunning MLMC with OT...")
c_mlmc_ot = make_c_OT(x0, T, h0, r, cov_mat, vol, max_deg, P1, s_min0, s_max0, C)
b_bar_mlmc_ot = make_b_bar(c_mlmc_ot, pairs, s_min0, s_max0, T, max_deg)

plot_volatility_surface(b_bar_mlmc_ot, s_min0, s_max0, T,
                        title=f"OT-MLMC Volatility Surface (max_deg={max_deg})",
                        save_path=f"plots/VolSurfOT/OTVolSurf_maxdeg{max_deg}.pdf")

---

## 3. Fine/Coarse Coupling Diagnostics

**Based on `pairsscatter.py`**

Visualises correlation between fine and coarse paths.

In [ ]:
def plot_pairwise_scatter(paths_f, paths_c, level, title_suffix="", save_path=None):
    """Create scatter plot of fine vs coarse at terminal time."""
    d = paths_f.shape[2]
    fig, axes = plt.subplots(1, d, figsize=(5*d, 4))
    if d == 1:
        axes = [axes]
    
    for i, ax in enumerate(axes):
        X_f = paths_f[:, -1, i]
        X_c = paths_c[:, -1, i]
        
        ax.scatter(X_c, X_f, alpha=0.4, s=5, color='steelblue')
        x_range = [min(X_c.min(), X_f.min()), max(X_c.max(), X_f.max())]
        ax.plot(x_range, x_range, 'r--', linewidth=2, label='Perfect correlation')
        
        corr = np.corrcoef(X_c, X_f)[0, 1]
        ax.set_xlabel(f'Coarse $X^c_{i+1}(T)$', fontsize=11)
        ax.set_ylabel(f'Fine $X^f_{i+1}(T)$', fontsize=11)
        ax.set_title(f'Asset {i+1}: ρ = {corr:.4f}', fontsize=12)
        ax.grid(True, alpha=0.3)
        ax.legend()
    
    fig.suptitle(f'Level {level} Fine/Coarse Correlation {title_suffix}', fontsize=14, y=1.02)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
        print(f"✓ Saved: {save_path}")
    plt.show()

In [ ]:
print("\nGenerating OT maps and paths...")
l_test = 1
maps, paths_f_red, paths_c_ot = logpaths_maps(
    x0, T, h0, l_test, r, cov_mat, vol, max_deg,
    return_redpathsf=True, return_pathsc=True, C=40
)
est_paths_c_ot = apply_maps(paths_f_red, maps, input_logpaths=False)

plot_pairwise_scatter(paths_f_red, est_paths_c_ot, l_test,
                      title_suffix="(OT)",
                      save_path=f"plots/PairScatter/PairDistribution_Level{l_test}_OT.pdf")

---

## 4. Summary

In [ ]:
print("\n" + "="*70)
print("RESULTS SUMMARY")
print("="*70)
print("✓ Volatility surfaces generated")
print("✓ Coupling quality analysed")
print("✓ All plots saved to plots/ directory")
print("="*70)